# CRISPRtracrRNA Prediction

This notebook demonstrates `run_crispr_tracr_rna`, which scans nucleotide sequences for tracrRNA candidates using the CRISPRtracrRNA tool from the Backofen Lab. tracrRNA is required by type II CRISPR-Cas9 systems to form the guide RNA complex; identifying it is a critical step when characterizing novel Cas9 orthologs or validating candidate loci in a CRISPR engineering pipeline.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("crispr_tracr_rna")
display_overview("crispr_tracr_rna")
display_docs_section("crispr_tracr_rna", "Background")

# CRISPRtracrRNA

> CRISPRtracrRNA predicts [tracrRNA](https://en.wikipedia.org/wiki/Trans-activating_crRNA) (trans-activating CRISPR RNA) sequences from nucleotide [CRISPR](https://en.wikipedia.org/wiki/CRISPR) loci using [covariance models](http://eddylab.org/infernal/) and RNA-RNA interaction prediction. It identifies tracrRNA candidates, scores them by E-value and anti-repeat similarity, and predicts the anti-repeat interaction structure via [IntaRNA](https://rna.informatik.uni-freiburg.de/IntaRNA/).

## Background

**What does this tool measure/predict?**
This tool predicts the location and quality of tracrRNA sequences within CRISPR loci. The tracrRNA is a non-coding RNA essential for [Type II CRISPR-Cas](https://en.wikipedia.org/wiki/CRISPR#Class_2) systems; it base-pairs with the CRISPR repeat (via its anti-repeat region) and forms a complex with [Cas9](https://en.wikipedia.org/wiki/Cas9) to guide DNA cleavage.

**Why is this important?**
For a CRISPR-Cas9 system to function, three components must be present: the Cas9 protein, the [crRNA](https://en.wikipedia.org/wiki/CRISPR#crRNA) (from the CRISPR array), and the tracrRNA. Detecting the tracrRNA is essential for:

* Confirming that a CRISPR locus encodes a complete, functional Type II system
* Designing [single-guide RNAs](https://en.wikipedia.org/wiki/Guide_RNA) (sgRNAs) by fusing crRNA and tracrRNA sequences
* Validating generated CRISPR systems (e.g., from Evo1) for functional completeness
* Discovering novel CRISPR-Cas9 systems in metagenomic data

**Scientific foundation:**
CRISPRtracrRNA uses [Infernal](http://eddylab.org/infernal/) covariance models (CMs) to detect tracrRNA candidates. Covariance models are probabilistic models of RNA sequence and secondary structure; similar to [profile HMMs](https://en.wikipedia.org/wiki/Hidden_Markov_model) but extended to capture base-pairing (covariance). After identifying tracrRNA candidates, the tool uses [IntaRNA](https://rna.informatik.uni-freiburg.de/IntaRNA/) to predict the RNA-RNA interaction between the tracrRNA's anti-repeat region and the CRISPR repeat, providing a structural validation of the prediction.

## Available tools

In [2]:
display_available_tools("crispr_tracr_rna")

- **`run_crispr_tracr_rna()`** — Predict tracrRNA sequences from nucleotide CRISPR loci

### `run_crispr_tracr_rna`

`run_crispr_tracr_rna` predicts tracrRNA sequences from nucleotide sequences that contain CRISPR loci. The tool evaluates anti-repeat complementarity and uses IntaRNA to compute RNA-RNA interaction energies, returning predicted tracrRNA coordinates and interaction energy scores for each input sequence. In practice the input should be a confirmed CRISPR-containing genomic region, typically identified by a tool like MinCED in an earlier analysis step.

In [3]:
from proto_tools import (
    CrisprTracrRNAInput,
    CrisprTracrRNAConfig,
    run_crispr_tracr_rna,
)

In [4]:
# Display input docs
display_api_reference("crispr_tracr_rna", "input", "run_crispr_tracr_rna")

# Provide a genomic sequence containing a CRISPR locus
# In practice, use a real CRISPR-containing genomic region
inputs = CrisprTracrRNAInput(
    sequences=["ATGCGTAAACGATTGCAGT" * 500],
    sequence_ids=["locus_1"],
)

**Input** — `CrisprTracrRNAInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `List[string]` | required | Nucleotide sequence(s) to predict tracrRNA from. Each sequence should contain a CRISPR locus. |
| `sequence_ids` | `array` |  | Optional sequence identifiers. |

In [5]:
# Display config docs
display_api_reference("crispr_tracr_rna", "config", "run_crispr_tracr_rna")

# Type II model for Cas9-associated tracrRNA prediction | see docs above for all fields
config = CrisprTracrRNAConfig(model_type="II")

**Config** — `CrisprTracrRNAConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `model_type` | `enum` | `II` | Type of CRISPR model to use. Available options: `II`, `all` |
| `run_type` | `enum` | `complete_run` | Pipeline mode (complete\_run or model\_only). Available options: `complete_run`, `model_run` |
| `num_workers` | `integer` |  | Number of parallel workers. |
| `seed` | `integer` |  | Random seed for reproducible results. When set, same seed + same input produces identical output. When None, a random seed is auto-assigned internally via the `resolved_seed` property. Some GPU tools produce approximately (not bit-exact) identical results because CUDA operations reduce floating-point values in non-deterministic thread order. Discrete outputs (sequences, structures) are exact; floating-point outputs (scores, coordinates) may differ at the bit level (\~1e-7). |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cpu` | Device to run the tool on. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [6]:
# Run the tracrRNA prediction
result = run_crispr_tracr_rna(inputs, config)

Running run_crispr_tracr_rna [00:00]

In [ ]:
# Display output docs
display_api_reference("crispr_tracr_rna", "output", "run_crispr_tracr_rna")

# One CrisprTracrRNASequenceResult per input sequence; each carries every candidate hit
# upstream produced for that accession, sorted by score descending.
for seq_result in result.results:
    top = seq_result.top_candidate
    if top is not None and top.has_tracr:
        print(f"{seq_result.sequence_id}: tracrRNA at {top.anti_repeat_start}-{top.anti_repeat_end} "
              f"(score={top.score}, {len(seq_result.candidates)} candidates)")
        print(f"  Interaction energy: {top.interaction_energy} kcal/mol")
    else:
        print(f"{seq_result.sequence_id}: no tracrRNA detected")

## Export Results

TracrRNA predictions can be exported to CSV or JSON. Both formats flatten to one row per candidate hit — sequences with multiple candidates produce multiple rows (sorted by `score` descending), and sequences with no candidates produce a single `sequence_id`-only row.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="crispr_tracr_rna_results", export_path=output_dir, file_format="json")
print(f"tracrRNA predictions exported to {output_dir}")